In [8]:
print("ok")

ok


In [9]:
import os
os.chdir("../")
%cd /Users/primeshofficial/Documents/MedicalBOT/MedicalChatbot

/Users/primeshofficial/Documents/MedicalBOT/MedicalChatbot


In [10]:
%pwd

'/Users/primeshofficial/Documents/MedicalBOT/MedicalChatbot'

In [41]:
from langchain.document_loaders import PyPDFLoader , DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter  


In [44]:
#exact text from pdf
def load_pdf_files(data):
    loader = DirectoryLoader(
        data, 
        glob="*.pdf", 
        loader_cls=PyPDFLoader )
    document = loader.load()
    return document

In [45]:
extracted_data = load_pdf_files("data")

Ignoring wrong pointing object 34 0 (offset 0)
Ignoring wrong pointing object 36 0 (offset 0)
Ignoring wrong pointing object 56 0 (offset 0)
Ignoring wrong pointing object 148 0 (offset 0)
Ignoring wrong pointing object 6 0 (offset 0)
Ignoring wrong pointing object 14 0 (offset 0)
Ignoring wrong pointing object 15 0 (offset 0)
Ignoring wrong pointing object 33 0 (offset 0)
Ignoring wrong pointing object 74 0 (offset 0)
Ignoring wrong pointing object 409 0 (offset 0)


In [46]:
from typing import List
from langchain.schema import Document

def filter_to_minimal_docs(docs: List[Document]) -> List[Document]:
    # Implement your logic to filter and return minimal documents
    minimal_docs :list[Document] = []
    for doc in docs:
        src=doc.metadata.get("source")
        minimal_docs.append(
            Document(
                page_content=doc.page_content,
                metadata={"source": src}
            )
        )
    return minimal_docs

In [47]:
minimal_docs = filter_to_minimal_docs(extracted_data)

In [48]:
minimal_docs

[Document(metadata={'source': 'data/Current Essentials of Medicine(1)(1).pdf'}, page_content=''),
 Document(metadata={'source': 'data/Current Essentials of Medicine(1)(1).pdf'}, page_content='a LANGE medical book\nCURRENT\nESSENTIALS\nof MEDICINE\nFourth Edition\nEdited by\nLawrence M. Tierney, Jr., MD\nProfessor of Medicine\nUniversity of California, San Francisco\nAssociate Chief of Medical Services\nVeterans Affairs Medical Center\nSan Francisco, California\nSanjay Saint, MD, MPH\nAssociate Chief of Medicine, Ann Arbor VA Medical Center\nDirector, VA/UM Patient Safety Enhancement Program\nProfessor of Internal Medicine, University of Michigan Medical School \nAnn Arbor, Michigan\nMary A. Whooley, MD\nProfessor of Medicine, Epidemiology and Biostatistics \nUniversity of California, San Francisco\nDepartment of Veterans Affairs Medical Center\nSan Francisco, California\nNew York Chicago San Francisco Lisbon London Madrid Mexico City\nMilan New Delhi San Juan Seoul Singapore Sydney Tor

In [52]:
#spliit the do in to smaller chunks
def text_splitter(minimal_docs):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=20
    )
    texts_chunks=text_splitter.split_documents(minimal_docs)
    return texts_chunks

In [53]:
texts_chunks = text_splitter(minimal_docs)
print(f"Number of text chunks: {len(texts_chunks)}")

Number of text chunks: 8441


In [54]:
from langchain.embeddings import HuggingFaceEmbeddings

def download_embeddings():
    model_name = 'sentence-transformers/all-MiniLM-L6-v2'
    embeddings = HuggingFaceEmbeddings(
        model_name=model_name,
    )
    return embeddings

embedding = download_embeddings()

In [55]:
embedding


HuggingFaceEmbeddings(client=SentenceTransformer(
  (0): Transformer({'max_seq_length': 256, 'do_lower_case': False}) with Transformer model: BertModel 
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
), model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, multi_process=False, show_progress=False)

In [56]:
from dotenv import load_dotenv
import os
load_dotenv()


True

In [57]:
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
os.environ["GROQ_API_KEY"] = GROQ_API_KEY
os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY

In [58]:
from pinecone import Pinecone
pinecone_api_key = PINECONE_API_KEY
pc = Pinecone(api_key=pinecone_api_key)

In [59]:
pc

In [60]:
from pinecone import ServerlessSpec
index_name = "medicalbot"
if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=384,  # Dimension of the embedding vectors
        metric="cosine",  # Similarity metric (e.g., "cosine", "euclidean")
        spec=ServerlessSpec(cloud="aws", region="us-east-1")  # Serverless configuration
    )
index = pc.Index(index_name)

In [61]:
from langchain_pinecone import PineconeVectorStore
docsearch = PineconeVectorStore.from_documents(
    documents=texts_chunks,
    embedding=embedding,
    index_name=index_name
)

In [62]:
#load exisiting details from pinecone
from langchain_pinecone import PineconeVectorStore
docsearch = PineconeVectorStore.from_existing_index(
    embedding=embedding,
    index_name=index_name
)

# Add more docs if you want

In [ ]:
#dummydocs
dswith=Document(
    page_content="This is a dummy document for testing.",
    metadata={"source": "dummy_source"})

In [ ]:
docsearch.add_documents([dswith])

['57233511-bd5d-48c1-883b-a18e8db13a2b']

In [71]:
retriever = docsearch.as_retriever(search_type="similarity",search_kwargs={"k": 1000})

In [72]:
retrieved_docs = retriever.invoke("What is acne?")
retrieved_docs

[Document(id='e48a3984-d6a6-4743-a477-7f9c9a099cc7', metadata={'source': 'data/Medical_book.pdf'}, page_content='GALE ENCYCLOPEDIA OF MEDICINE 226\nAcne\nGEM - 0001 to 0432 - A  10/22/03 1:41 PM  Page 26'),
 Document(id='0d894c5f-7b74-4392-a064-d455da378739', metadata={'source': 'data/Medical_book.pdf'}, page_content='GALE ENCYCLOPEDIA OF MEDICINE 226\nAcne\nGEM - 0001 to 0432 - A  10/22/03 1:41 PM  Page 26'),
 Document(id='76957603-fe13-4b80-b461-7752f9851f5b', metadata={'source': 'data/Medical_book.pdf'}, page_content='GALE ENCYCLOPEDIA OF MEDICINE 226\nAcne\nGEM - 0001 to 0432 - A  10/22/03 1:41 PM  Page 26'),
 Document(id='4b3ec63e-2cb4-4338-a047-a43be8547853', metadata={'source': 'data/Medical_book.pdf'}, page_content='GALE ENCYCLOPEDIA OF MEDICINE 226\nAcne\nGEM - 0001 to 0432 - A  10/22/03 1:41 PM  Page 26'),
 Document(id='fa6f89a4-c5d0-4e98-b5fa-427132c5761a', metadata={'source': 'data/Medical_book.pdf'}, page_content='GALE ENCYCLOPEDIA OF MEDICINE 2 25\nAcne\nAcne vulgaris aff

In [33]:
from langchain_groq import ChatGroq
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_openai import ChatOpenAI
chat_model1 = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key= os.getenv("GROQ_API_KEY"),
    temperature=0
)


chat_model3 = ChatOpenAI(
    model="gpt-4o",
    api_key=os.getenv("OPENAI_API_KEY"),
    temperature=0
)
chat_model2 = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",   # free & fast
    google_api_key=os.getenv("GEMINI_API_KEY"),
    temperature=0
)


/opt/anaconda3/envs/medibotenv/lib/python3.11/site-packages/langchain_google_genai/chat_models.py:47: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  from google.generativeai.caching import CachedContent  # type: ignore[import]


In [34]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.prompts import ChatPromptTemplate

In [35]:
system_prompt = (
    "You are an Medical assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the "
    "answer concise."
    "\n\n"
    "{context}"
)
prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}")
])

In [36]:
question_answering_chain = create_stuff_documents_chain(chat_model1, prompt)
rag_chain = create_retrieval_chain(retriever, question_answering_chain)

In [37]:
import time
start = time.time()
retrieved_docs = retriever.invoke("What is diabetes?")

print("Retrieval:", time.time() - start)

start = time.time()
response = rag_chain.invoke({"input": "What is diabetes?"})

print("Total:", time.time() - start)

Retrieval: 1.753061056137085
Total: 1.2476048469543457


In [38]:
import time

start = time.time()

response = chat_model1.invoke("What is diabetes?")

print(time.time() - start)
print(response.content)

1.3947160243988037
Diabetes is a chronic health condition that affects the way your body turns food into energy. It's a disorder of the metabolism, which is the process by which your body converts the food you eat into energy.

When you eat, your body breaks down carbohydrates into a type of sugar called glucose. Glucose is then absorbed into your bloodstream, where it's carried to your cells. Insulin, a hormone produced by the pancreas, helps your cells absorb glucose from the bloodstream.

In people with diabetes, the body either:

1. **Can't produce enough insulin** (Type 1 diabetes): The pancreas doesn't produce enough insulin, so glucose builds up in the bloodstream.
2. **Can't effectively use insulin** (Type 2 diabetes): The body becomes resistant to insulin, making it harder for glucose to enter the cells.
3. **Has a combination of both** (LADA, or Latent Autoimmune Diabetes in Adults): The body produces some insulin, but not enough, and also has insulin resistance.

As a result

In [73]:
response = rag_chain.invoke({"input": "what is skin cancer?"})
print(response["answer"])

Skin cancer is one of the most common cancers, which can be divided into non-melanoma (basal cell carcinoma and squamous cell carcinoma) and melanoma (malignant melanoma). It is a type of cancer that affects the skin, with risk factors including excessive UV exposure and genetic predisposition. Malignant melanoma is the most life-threatening type of skin cancer, often affecting younger populations.
